# High-Throughput Batch PDF -> Structured JSON Extraction via Datalab Lift (Google Colab T4 Edition)

This notebook processes scanned sales order info sheet PDFs (e.g. `Batch3`) using **Datalab Lift VLM** directly into structured JSON conforming to the **ACE Infosheet** production schema.

### Active Performance Optimizations:
1. **Compact JSON Schema in Prompt**: Minified schema eliminates ~900 whitespace tokens per forward pass.
2. **Continuation Chunk Schema Pruning**: Pages after chunk 0 only prompt for `special_instructions` using `ContinuationExtraction`, omitting the repeated `Header` schema.
3. **Pipelined CPU-GPU Prefetching**: A lightweight background daemon thread (`BackgroundPDFLoader`) relocates and rasterizes the *next* PDF while the GPU is processing the *current* PDF, completely eliminating GPU idle wait times. Memory-bounded (`maxsize=1`, < 20 MB RAM).
4. **Tesla T4 Hardware FP16 Acceleration**: Enforces native FP16 Tensor Cores (65 TFLOPS) avoiding 3x-4x slow emulation from bfloat16.
5. **CuDNN Benchmark & Inference Mode**: `torch.inference_mode()` and `torch.backends.cudnn.benchmark = True` optimize kernel selection and bypass autograd overhead.
6. **One-Time Google Drive 4-Bit Model Caching**: Saves the 4-bit quantized model weights (~4.8 GB) directly to `MyDrive/lift_model_4bit`. Subsequent sessions load in ~30 seconds, skipping the 19.3 GB Hugging Face download entirely.

---
### Prerequisites:
1. **Google Colab Runtime**: **Runtime** -> **Change runtime type** -> **T4 GPU** (Free tier) -> **Save**.
2. **Google Drive Preparation**: Upload your `Batch3` folder (or `Batch_3`) containing PDFs to your Google Drive (e.g. `My Drive/Batch3`).
3. **Checkpoint Database**: Progress is checkpointed in a persistent SQLite database on Google Drive (`pipeline_state_colab.sqlite3`) so you can safely interrupt and resume at any time.


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("Google Drive mounted successfully.")


In [ ]:
# Step 2: GPU VRAM Cleanup & Memory Reset Utility
# Run this cell anytime you want to clear out lingering models/tensors or free up VRAM
import gc
import torch

def cleanup_gpu():
    """Deletes lingering model/tensor/loader variables, runs garbage collection, and empties CUDA cache."""
    # Stop background loader if running
    if "loader" in globals():
        try:
            globals()["loader"].stop()
        except Exception:
            pass

    # Clean up global references if they exist from prior cell executions
    for var in ["extractor", "model", "processor", "inputs", "output_ids", "extraction", "images", "loader"]:
        if var in globals():
            try:
                del globals()[var]
            except Exception:
                pass

    # Force multi-pass Python garbage collection
    gc.collect()
    gc.collect()

    # Flush PyTorch CUDA memory allocator caches
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()
        allocated = torch.cuda.memory_allocated() / (1024**3)
        reserved = torch.cuda.memory_reserved() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        free = total - reserved
        print("GPU VRAM Cleanup Complete:")
        print(f"  Active Tensor Memory: {allocated:.2f} GB")
        print(f"  PyTorch Reserved:     {reserved:.2f} GB")
        print(f"  Available VRAM:       {free:.2f} GB / {total:.1f} GB")
    else:
        print("CUDA is not active yet.")

cleanup_gpu()


In [ ]:
# Step 3: Install required production libraries
!pip install -q transformers accelerate bitsandbytes pypdfium2 pydantic rich
print("Required libraries installed.")


In [ ]:
# Step 4: Verify GPU hardware and configure optimal compute dtype for T4
import torch

if not torch.cuda.is_available():
    raise SystemExit("ERROR: GPU runtime not found! Please enable T4 GPU via: Runtime -> Change runtime type -> T4 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
major_cc, minor_cc = torch.cuda.get_device_capability()

print(f"Detected GPU: {device_name} ({vram_gb:.1f} GB VRAM, Compute Capability {major_cc}.{minor_cc})")

# TESLA T4 HARDWARE REALITY:
# Tesla T4 (Turing, compute capability 7.5) has native hardware Tensor Cores for float16 (65 TFLOPS),
# but NO hardware Tensor Cores for bfloat16.
# If bfloat16 is selected on T4, PyTorch runs in software emulation which is 3x-4x SLOWER!
# We strictly use float16 on T4, reserving bfloat16 for Compute Capability >= 8.0 (A100/L4).
COMPUTE_DTYPE = torch.bfloat16 if major_cc >= 8 else torch.float16
print(f"Tensor Core compute dtype: {COMPUTE_DTYPE}")

# CuDNN kernel auto-tuning
torch.backends.cudnn.benchmark = True
print("cuDNN benchmark auto-tuner enabled.")


In [ ]:
# Step 5: Configure paths, model cache, and batch settings
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

possible_source = DRIVE_ROOT / "Batch_3"

SOURCE_DIR = possible_source
OUTPUT_DATA_DIR = DRIVE_ROOT / "data"

# Checkpoint database stored on Drive for full session persistence
STATE_DB_PATH = DRIVE_ROOT / "pipeline_state_colab.sqlite3"

# 4-bit Quantized Model Cache paths:
# Cache on Google Drive persists across Colab sessions (~4.8 GB vs 19.3 GB raw download)
LOCAL_MODEL_DIR = Path("/content/lift_model_4bit")
DRIVE_MODEL_DIR = DRIVE_ROOT / "lift_model_4bit"
SAVE_MODEL_TO_DRIVE = True

# Batch execution settings:
CHUNK_SIZE = 7
DPI = 72
DEFAULT_MAX_TOKENS = 16384
STOP_ON_ERROR = False  # Keep going if one file fails; log and continue

print(f"Source PDF folder:  {SOURCE_DIR} (exists: {SOURCE_DIR.exists()})")
print(f"Target data folder: {OUTPUT_DATA_DIR}")
print(f"State DB file:      {STATE_DB_PATH}")
print(f"Local model cache:  {LOCAL_MODEL_DIR}")
print(f"Drive model cache:  {DRIVE_MODEL_DIR} (exists: {DRIVE_MODEL_DIR.exists()})")


In [ ]:
# Step 6: Define the ACE Infosheet Extraction Pydantic Schemas
import json
from typing import Optional, List, Any
from pydantic import BaseModel, Field

class Header(BaseModel):
    customer_name: Optional[str] = Field(None, description="Customer company name")
    customer_bp_no: Optional[str] = Field(None, description="Customer BP number")
    customer_address: Optional[str] = Field(None, description="Complete customer address")
    plant: Optional[str] = Field(None, description="Plant code")
    customer_so_no: Optional[str] = Field(None, description="Customer sales order number")
    crm_order_id: Optional[str] = Field(None, description="CRM order ID")
    so_type: Optional[str] = Field(None, description="Sales order type")
    so_approval_date: Optional[str] = Field(None, description="SO approval date, YYYY-MM-DD")
    internal_so_no: Optional[str] = Field(None, description="Internal sales order number")
    po_number: Optional[str] = Field(None, description="Purchase order number")
    po_date: Optional[str] = Field(None, description="PO date, YYYY-MM-DD")
    material_id: Optional[str] = Field(None, description="Material ID")
    material_description: Optional[str] = Field(None, description="Material description")
    required_delivery_date: Optional[str] = Field(None, description="Required delivery date, YYYY-MM-DD")
    quantity: Optional[int] = Field(None, description="Required quantity")
    machine_serial_no: Optional[str] = Field(None, description="Machine serial number")

class SpecialInstruction(BaseModel):
    sl_no: int = Field(..., description="Original printed serial number")
    instruction: str = Field(..., description="Instruction copied faithfully from document")
    is_amendment: bool = Field(..., description="True for EARLIER or AMENDED AS rows")

# Full schema for chunk 0 (extracts Header + Special Instructions)
class Extraction(BaseModel):
    header: Header
    special_instructions: list[SpecialInstruction]

# Optimization 2: Continuation schema for subsequent chunks (extracts Special Instructions ONLY)
class ContinuationExtraction(BaseModel):
    special_instructions: list[SpecialInstruction]

print("Extraction schemas loaded (including pruned ContinuationExtraction schema).")


In [ ]:
# Step 7: System Prompt and Extraction Cleaning Filters
import pypdfium2 as pdfium
from PIL import Image

MODEL_ID = "datalab-to/lift"

LIFT_SYSTEM_PROMPT = (
    "You are a specialized document data extraction engine for industrial CNC machine sales order infosheets.\n"
    "Extract structured data from this document strictly adhering to the target JSON schema.\n\n"
    "DOCUMENT LAYOUT (CRITICAL - READ CAREFULLY):\n"
    "Every infosheet has a 'Scope of Supply' table that is divided into TWO zones by a special divider row:\n\n"
    "  ZONE 1 - STANDARD MACHINE SPECIFICATIONS (DO NOT EXTRACT AS SPECIAL INSTRUCTIONS):\n"
    "    These are fixed/templated rows that appear in every order for a given machine model.\n"
    "    They typically occupy Sl.No 1 through approximately Sl.No 30-37 and cover items such as:\n"
    "    Machine Model, Machine market, Bed, Spindle Type, Spindle speed, Package,\n"
    "    Axis Assembly, Tooling type, Tailstock, Work Holding, PCC make, coolant&chip asy,\n"
    "    Lubrication system, Hydraulic power pack, Guarding, TPM requirement, Cabinet cooling,\n"
    "    Transformer type, Machine colour, Packing, Spindle orientation, Axes calibration,\n"
    "    Electronic hand wheel, Thro spindle fluid, Torque enhancement, Oil separation method,\n"
    "    Tool life management, Trials, Machine manuals, Training, Hydraulic oil cooling,\n"
    "    Door interlock, TPM TRAK, Special (pointing to 'Refer Special Text'), CNC System.\n\n"
    "  DIVIDER ROW (DO NOT EXTRACT):\n"
    "    A row whose 'Scope of Supply' cell contains exactly 'Special requirement/Instructions'\n"
    "    or a close variant. This row has no Sl.No or the Sl.No cell is blank.\n"
    "    It marks the boundary between Zone 1 and Zone 2.\n\n"
    "  ZONE 2 - ACTUAL SPECIAL INSTRUCTIONS (EXTRACT ALL OF THESE):\n"
    "    Every row AFTER the 'Special requirement/Instructions' divider row is a customer-specific\n"
    "    special instruction. These typically start at Sl.No 31-38 on page 1 and continue on\n"
    "    all subsequent pages (pages 2, 3, 4, etc. of multi-page documents).\n"
    "    On continuation pages the table header row repeats but ALL data rows are special instructions.\n"
    "    Extract every Zone 2 row faithfully.\n\n"
    "EXTRACTION RULES:\n"
    "1. HEADER: Extract order and customer metadata. customer_name is in the 'Customer (BP No: ...)' block, "
    "NEVER the top masthead company name ('Ace Designers Limited' or similar).\n"
    "2. SPECIAL INSTRUCTIONS: Copy every Zone 2 instruction faithfully - do not summarize, correct "
    "spelling, or merge rows. Preserve the original printed Sl.No. Mark EARLIER / AMENDED AS rows "
    "with is_amendment=true; all others get is_amendment=false.\n"
    "3. NEVER extract Zone 1 rows (Sl.No 1 through the divider) as special instructions.\n"
    "4. NEVER extract footer text or approval/signatory tables (Prepared By, Checked By, Approved By, "
    "Field Representative, Authorised Signatory, contact numbers).\n"
    "5. On multi-page documents the masthead ('Ace Designers Limited...') and sub-header "
    "('Customer So No / Int So No') repeat on every page - ignore these repeated headers.\n"
    "6. Return only valid JSON conforming strictly to the schema."
)

def rasterize_pdf(pdf_path: Path, image_dpi: int = DPI) -> List[Image.Image]:
    doc = pdfium.PdfDocument(str(pdf_path))
    images: List[Image.Image] = []
    for page_idx in range(len(doc)):
        page_obj = doc[page_idx]
        pil_image = page_obj.render(scale=image_dpi / 72.0).to_pil().convert("RGB")
        images.append(pil_image)
    doc.close()
    return images

def clean_extracted_instructions(raw_instructions: list[dict[str, Any]]) -> list[dict[str, Any]]:
    ZONE1_PREFIXES = (
        "ace cnc lathe model", "machine model", "machine market",
        "bed:", "bed ", "spindle type", "spindle speed", "spindle orientation",
        "package:", "package ", "axis assembly", "axes assy", "axes assembly",
        "tooling type", "tailstock", "add on live centre", "add-on live centre",
        "work holding:", "work holding ", "make of chuck", "pcc make",
        "coolant&chip", "coolant & chip", "coolant filtration", "lubrication system",
        "hydraulic power pack", "guarding:", "tpm requirement", "cabinet cooling",
        "transformer type", "machine colour", "machine color", "packing:",
        "axes calibration", "electronic hand wheel", "thro spindle fluid",
        "through spindle fluid", "torque enhancement", "oil separation method",
        "tool life management", "machine manuals", "training:",
        "hydraulic oil cooling", "door interlock", "tpm trak", "cnc system",
        "standard machine specification", "ace cnc lathe",
    )
    EXCLUDE_KEYWORDS = (
        "prepared by", "checked by", "approved by", "field representative",
        "authorised signatory", "authorized signatory", "designation:",
        "sign & date", "sign and date", "customer seal & sign",
    )
    ZONE1_SL_NO_MAX = 37
    cleaned = []

    for item in raw_instructions:
        if not isinstance(item, dict):
            continue
        inst_text = str(item.get("instruction") or item.get("Scope of Supply") or item.get("text") or "").strip()
        spec_text = str(item.get("Specification") or item.get("specification") or "").strip()
        if spec_text and spec_text.lower() not in ("none", "null", "nil", "-", "") and spec_text.lower() not in inst_text.lower():
            inst_text = f"{inst_text} - {spec_text}" if inst_text else spec_text
        if not inst_text:
            continue
        lower = inst_text.lower()
        if any(kw in lower for kw in EXCLUDE_KEYWORDS):
            continue
        if lower in ("special requirement/instructions", "special requirement / instructions", "special requirements/instructions"):
            continue
        if any(lower.startswith(p) for p in ZONE1_PREFIXES):
            continue
        for prefix_variant in ("special requirement/instructions:", "special instructions:", "special requirement:"):
            if lower.startswith(prefix_variant):
                inst_text = inst_text.split(":", 1)[1].strip()
                lower = inst_text.lower()
                break
        if not inst_text or any(lower.startswith(p) for p in ZONE1_PREFIXES):
            continue
        orig_sl = item.get("sl_no")
        if orig_sl is not None:
            try:
                if int(orig_sl) <= ZONE1_SL_NO_MAX:
                    zone1_kws = ("machine model", "bed", "spindle", "axis", "tooling", "tailstock", "coolant", "lubrication", "hydraulic", "guarding", "cnc system")
                    if any(kw in lower for kw in zone1_kws):
                        continue
            except (ValueError, TypeError):
                pass
        is_amend = bool(item.get("is_amendment", False))
        if "amendment" in lower or "earlier:" in lower or "amended as:" in lower:
            is_amend = True
        cleaned.append({
            "sl_no": orig_sl if orig_sl is not None else len(cleaned) + 1,
            "instruction": inst_text,
            "is_amendment": is_amend,
        })
    return cleaned


In [ ]:
# Step 8: LiftExtractor Model Loader and Optimized Inference Engine
import gc
import time
import os
import shutil
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

class LiftExtractorColab:
    def __init__(self, chunk_size: int = CHUNK_SIZE):
        self.chunk_size = chunk_size
        self.device = "cuda:0"

        # Determine model source: Local SSD -> Google Drive -> Hugging Face download
        is_fresh_hf_download = False
        if LOCAL_MODEL_DIR.exists() and (LOCAL_MODEL_DIR / "config.json").exists():
            print(f"[LiftExtractor] Found cached 4-bit model on local SSD: {LOCAL_MODEL_DIR}")
            model_source = str(LOCAL_MODEL_DIR)
        elif DRIVE_MODEL_DIR.exists() and (DRIVE_MODEL_DIR / "config.json").exists():
            print(f"[LiftExtractor] Found cached 4-bit model on Google Drive: {DRIVE_MODEL_DIR}")
            print(f"[LiftExtractor] Copying to fast local SSD ({LOCAL_MODEL_DIR}) to avoid Drive FUSE latency...")
            t_copy = time.time()
            if LOCAL_MODEL_DIR.exists():
                shutil.rmtree(LOCAL_MODEL_DIR)
            shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
            print(f"[LiftExtractor] Copied model to local SSD in {time.time() - t_copy:.1f}s.")
            model_source = str(LOCAL_MODEL_DIR)
        else:
            print(f"[LiftExtractor] No cached 4-bit model found. Initial download from Hugging Face '{MODEL_ID}'...")
            model_source = MODEL_ID
            is_fresh_hf_download = True

        print(f"[LiftExtractor] Loading processor from '{model_source}'...")
        self.processor = AutoProcessor.from_pretrained(model_source)
        if hasattr(self.processor, "tokenizer") and self.processor.tokenizer:
            self.processor.tokenizer.padding_side = "left"

        print(f"[LiftExtractor] Loading model into VRAM (compute_dtype={COMPUTE_DTYPE})...")
        t0 = time.time()

        if is_fresh_hf_download:
            # On-the-fly 4-bit quantization with BitsAndBytes
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=COMPUTE_DTYPE,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
            self.model = AutoModelForImageTextToText.from_pretrained(
                model_source,
                quantization_config=quant_config,
                device_map={"": 0},
                torch_dtype=COMPUTE_DTYPE,
                attn_implementation="sdpa",
                low_cpu_mem_usage=True,
            )
            # Save 4-bit quantized weights to local SSD and Google Drive for future sessions
            if SAVE_MODEL_TO_DRIVE:
                print(f"[LiftExtractor] Exporting 4-bit quantized model to local SSD ({LOCAL_MODEL_DIR})...")
                print("  (Writing ~4.8 GB in 1GB shards; progress updates per shard...)")
                gc.collect()
                torch.cuda.empty_cache()
                self.model.save_pretrained(str(LOCAL_MODEL_DIR), max_shard_size="1GB", safe_serialization=True)
                self.processor.save_pretrained(str(LOCAL_MODEL_DIR))

                if DRIVE_ROOT.exists() and not (DRIVE_MODEL_DIR / "config.json").exists():
                    print(f"[LiftExtractor] Caching 4-bit model to Google Drive ({DRIVE_MODEL_DIR})...")
                    print("  (Copying ~4.8 GB over Google Drive FUSE; this takes ~2-3 mins. Please wait...)")
                    t_drv = time.time()
                    if DRIVE_MODEL_DIR.exists():
                        shutil.rmtree(DRIVE_MODEL_DIR)
                    shutil.copytree(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR)
                    print(f"[LiftExtractor] Model cached to Drive in {time.time() - t_drv:.1f}s (~4.8 GB). Future runs will skip HF download!")
        else:
            # Model is already 4-bit quantized; config.json carries quantization_config automatically
            self.model = AutoModelForImageTextToText.from_pretrained(
                model_source,
                device_map={"": 0},
                torch_dtype=COMPUTE_DTYPE,
                attn_implementation="sdpa",
                low_cpu_mem_usage=True,
            )

        self.model.eval()
        print(f"[LiftExtractor] Model ready in {time.time() - t0:.1f}s.")

    def _clean_and_parse_json(self, raw_text: str) -> dict[str, Any]:
        text = raw_text.strip()
        if "```json" in text:
            text = text.split("```json", 1)[1].split("```", 1)[0].strip()
        elif "```" in text:
            text = text.split("```", 1)[1].split("```", 1)[0].strip()
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1:
            text = text[start : end + 1]
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            import re
            fixed = re.sub(r',\s*([}\]])', r'\1', text)
            return json.loads(fixed)

    def extract_from_images(
        self,
        images: List[Image.Image],
        schema: dict[str, Any],
        is_continuation: bool = False,
        max_output_tokens: int = DEFAULT_MAX_TOKENS
    ) -> tuple[dict[str, Any], int, float]:
        t_start = time.time()

        # Optimization 1: Compact JSON schema minification saves ~900 whitespace tokens per forward pass
        schema_json = json.dumps(schema, separators=(',', ':'), ensure_ascii=False)

        # Optimization 2: Tailored prompt for full extraction vs continuation chunks
        if not is_continuation:
            prompt_text = (
                f"Extract data from this sales order info sheet document.\n"
                f"Adhere strictly to the JSON Schema below:\n```json\n{schema_json}\n```\n\n"
                f"IMPORTANT DOMAIN LAYOUT:\n"
                f"- Page 1: Top section contains order header. Middle/bottom contains Scope of Supply table.\n"
                f"  Divider row has 'Special requirement/Instructions'. Rows AFTER the divider are Zone 2 special instructions.\n"
                f"- Continuation pages: All data rows are Zone 2 special instructions. Extract every row faithfully.\n"
            )
        else:
            prompt_text = (
                f"Extract continuation special instructions from these pages.\n"
                f"Adhere strictly to the JSON Schema below:\n```json\n{schema_json}\n```\n\n"
                f"IMPORTANT:\n"
                f"- All data rows on these continuation pages are Zone 2 special instructions.\n"
                f"- Extract every special instruction faithfully with its original printed Sl.No.\n"
                f"- Do NOT extract header or masthead information.\n"
            )

        content = [{"type": "image", "image": img} for img in images]
        content.append({"type": "text", "text": prompt_text})
        messages = [
            {"role": "system", "content": LIFT_SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ]
        text_input = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text_input], images=images, padding=True, return_tensors="pt")
        inputs = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

        # Tokenizer stopping tokens (Qwen2 / Lift <|im_end|> token ID = 248046)
        eos_token_id = self.processor.tokenizer.eos_token_id
        im_end_id = self.processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
        stopping_ids = [tid for tid in [eos_token_id, im_end_id] if tid is not None]

        input_len = inputs["input_ids"].shape[1]
        try:
            # Optimization 6: torch.inference_mode() optimizes CUDA memory and bypasses version tracking
            with torch.inference_mode():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=max_output_tokens,
                    do_sample=False,
                    temperature=None,
                    top_p=None,
                    eos_token_id=stopping_ids,
                    pad_token_id=self.processor.tokenizer.pad_token_id or eos_token_id,
                )
        finally:
            del inputs

        generated_ids = output_ids[0, input_len:]
        output_text = self.processor.tokenizer.decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
        parsed = self._clean_and_parse_json(output_text)
        return parsed, len(generated_ids), time.time() - t_start

    def extract_from_rendered_images(self, images: List[Image.Image], doc_name: str = "") -> tuple[Extraction, int, float]:
        """Extracts structured data from pre-rendered PIL images passed from background prefetcher."""
        n_pages = len(images)

        # Clear PyTorch transient cache before starting document
        gc.collect()
        torch.cuda.empty_cache()

        if n_pages <= self.chunk_size:
            raw_dict, total_tokens, latency = self.extract_from_images(
                images,
                Extraction.model_json_schema(),
                is_continuation=False
            )
        else:
            chunks = [images[i : i + self.chunk_size] for i in range(0, n_pages, self.chunk_size)]

            # Chunk 0: Extracts Header + initial Special Instructions
            raw_dict, total_tokens, latency = self.extract_from_images(
                chunks[0],
                Extraction.model_json_schema(),
                is_continuation=False
            )
            all_inst = list(raw_dict.get("special_instructions") or [])

            # Continuation chunks: Optimization 2 schema pruning (Special Instructions ONLY)
            for c_idx, chunk in enumerate(chunks[1:], start=1):
                gc.collect()
                torch.cuda.empty_cache()
                c_dict, c_tok, c_lat = self.extract_from_images(
                    chunk,
                    ContinuationExtraction.model_json_schema(),
                    is_continuation=True
                )
                all_inst.extend(c_dict.get("special_instructions") or [])
                total_tokens += c_tok
                latency += c_lat

            raw_dict["special_instructions"] = all_inst

        # Clean Zone 1 specs and normalize instructions
        raw_dict["special_instructions"] = clean_extracted_instructions(raw_dict.get("special_instructions") or [])
        validated = Extraction.model_validate(raw_dict)
        return validated, total_tokens, latency

    def extract_pdf(self, pdf_path: Path) -> tuple[Extraction, int, float]:
        """Helper to rasterize and extract a single PDF on demand."""
        images = rasterize_pdf(pdf_path, image_dpi=DPI)
        return self.extract_from_rendered_images(images, doc_name=pdf_path.stem)

print("LiftExtractorColab loaded.")


In [ ]:
# Step 9: Asynchronous CPU-GPU Pipelined Batch Processing
import shutil
import sqlite3
import time
import threading
import queue

def init_checkpoint_db(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(str(db_path), timeout=60.0)
    conn.row_factory = sqlite3.Row
    conn.execute("""
    CREATE TABLE IF NOT EXISTS files (
        path TEXT PRIMARY KEY,
        extract_status TEXT NOT NULL DEFAULT 'pending',
        extract_seconds REAL,
        extract_error TEXT,
        extract_updated_at TEXT
    );
    """)
    conn.commit()
    return conn

def relocate_into_data_dir(pdf_path: Path, data_root: Path) -> Path:
    doc_dir = data_root / pdf_path.stem
    doc_dir.mkdir(parents=True, exist_ok=True)
    dst = doc_dir / pdf_path.name
    if dst.resolve() != pdf_path.resolve() and not dst.exists():
        shutil.copy2(pdf_path, dst)
    return dst

# Optimization 3: CPU-GPU Asynchronous Prefetcher Daemon Thread
class BackgroundPDFLoader(threading.Thread):
    def __init__(self, to_process: list[Path], output_data_dir: Path, dpi: int = DPI):
        super().__init__(daemon=True)
        self.to_process = to_process
        self.output_data_dir = output_data_dir
        self.dpi = dpi
        # Bounded queue holds at most 1 pre-rendered document (<20MB RAM)
        self.queue = queue.Queue(maxsize=1)
        self.stopped = threading.Event()

    def stop(self):
        self.stopped.set()

    def run(self):
        for idx, pdf_path in enumerate(self.to_process, start=1):
            if self.stopped.is_set():
                break
            try:
                # 1. Relocate PDF into data/<stem>/<stem>.pdf
                dst_pdf = relocate_into_data_dir(pdf_path, self.output_data_dir)
                # 2. Rasterize PDF into PIL Images on CPU
                images = rasterize_pdf(dst_pdf, image_dpi=self.dpi)
                item = (idx, pdf_path, dst_pdf, images, None)
            except Exception as e:
                item = (idx, pdf_path, None, None, e)

            # Enqueue item; blocks if GPU is still processing previous item
            while not self.stopped.is_set():
                try:
                    self.queue.put(item, timeout=0.5)
                    break
                except queue.Full:
                    continue

        # Signal completion with sentinel
        while not self.stopped.is_set():
            try:
                self.queue.put(None, timeout=0.5)
                break
            except queue.Full:
                continue

# Initialize extractor
extractor = LiftExtractorColab(chunk_size=CHUNK_SIZE)

# Discover PDFs in SOURCE_DIR
pdf_files = sorted(list(SOURCE_DIR.rglob("*.pdf")) + list(SOURCE_DIR.rglob("*.PDF")))
# Filter out any files already inside data directory
pdf_files = [p for p in pdf_files if OUTPUT_DATA_DIR not in p.parents]

print(f"\nDiscovered {len(pdf_files)} PDF(s) under {SOURCE_DIR}")
if not pdf_files:
    print("No PDFs found. Please verify SOURCE_DIR path.")
else:
    conn = init_checkpoint_db(STATE_DB_PATH)
    stems = [p.stem for p in pdf_files]
    conn.executemany("INSERT OR IGNORE INTO files (path) VALUES (?)", [(s,) for s in stems])
    conn.commit()

    completed = set(row[0] for row in conn.execute("SELECT path FROM files WHERE extract_status = 'done'"))
    to_process = [p for p in pdf_files if p.stem not in completed]
    print(f"Total: {len(pdf_files)} | Already Done: {len(completed)} | Remaining: {len(to_process)}\n")

    succeeded = 0
    errors = 0

    if to_process:
        # Start background prefetcher daemon
        loader = BackgroundPDFLoader(to_process, OUTPUT_DATA_DIR, dpi=DPI)
        loader.start()
        print("[Pipelining] CPU background prefetcher thread started.\n")

        try:
            while True:
                # Wait for next pre-rendered document from queue
                item = loader.queue.get()
                if item is None:
                    # End of batch reached
                    break

                idx, pdf_path, dst_pdf, images, err = item
                stem = pdf_path.stem
                tag = f"[{idx}/{len(to_process)}] {stem[:50]}"
                t_start = time.time()
                print(f"Processing {tag}...")

                if err is not None:
                    errors += 1
                    err_msg = str(err)
                    print(f"  -> ERROR during prefetch: {err_msg[:200]}")
                    conn.execute(
                        "UPDATE files SET extract_status = 'error', extract_error = ?, extract_updated_at = datetime('now') WHERE path = ?",
                        (err_msg[:2000], stem)
                    )
                    conn.commit()
                    if STOP_ON_ERROR:
                        break
                    continue

                try:
                    # Run VLM extraction on pre-rendered images (GPU forward pass)
                    extraction, tokens, latency = extractor.extract_from_rendered_images(images, doc_name=stem)

                    # Save JSON in data/<stem>/<stem>.json
                    json_path = dst_pdf.parent / f"{stem}.json"
                    json_path.write_text(
                        json.dumps(extraction.model_dump(), indent=2, ensure_ascii=False) + "\n",
                        encoding="utf-8"
                    )

                    # Checkpoint success
                    conn.execute(
                        "UPDATE files SET extract_status = 'done', extract_seconds = ?, extract_error = NULL, extract_updated_at = datetime('now') WHERE path = ?",
                        (round(time.time() - t_start, 1), stem)
                    )
                    conn.commit()
                    succeeded += 1
                    print(f"  -> SUCCESS ({latency:.1f}s, {len(extraction.special_instructions)} instructions, {len(images)} pages)")

                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    errors += 1
                    err_msg = str(e)
                    print(f"  -> ERROR: {err_msg[:200]}")
                    conn.execute(
                        "UPDATE files SET extract_status = 'error', extract_error = ?, extract_updated_at = datetime('now') WHERE path = ?",
                        (err_msg[:2000], stem)
                    )
                    conn.commit()
                    if STOP_ON_ERROR:
                        break

        except KeyboardInterrupt:
            print("\n[WARN] Batch interrupted by user. Stopping background prefetcher...")
            loader.stop()
            # Drain queue to unblock loader thread
            while not loader.queue.empty():
                try:
                    loader.queue.get_nowait()
                except queue.Empty:
                    break
            print("Completed files remain checkpointed in SQLite database.")
        finally:
            loader.stop()

    conn.close()
    print(f"\n=== Batch Complete: {succeeded} succeeded, {errors} errors in this session. ===")


In [ ]:
# Step 10: Post-Processing Validation
# Verifies that every folder under data/ strictly adheres to folder structuring and naming conventions

doc_dirs = sorted([d for d in OUTPUT_DATA_DIR.iterdir() if d.is_dir()])
valid_count = 0
issues = []

for d in doc_dirs:
    stem = d.name
    pdf_candidates = list(d.glob(f"{stem}.pdf")) + list(d.glob(f"{stem}.PDF"))
    json_path = d / f"{stem}.json"

    if not pdf_candidates:
        issues.append(f"{stem}: Missing backing PDF file")
        continue
    if not json_path.is_file() or json_path.stat().st_size == 0:
        issues.append(f"{stem}: Missing or empty JSON file")
        continue

    try:
        data = json.loads(json_path.read_text(encoding='utf-8'))
        Extraction.model_validate(data)
        valid_count += 1
    except Exception as e:
        issues.append(f"{stem}: Invalid JSON schema: {e}")

print(f"Total Document Folders under {OUTPUT_DATA_DIR}: {len(doc_dirs)}")
print(f"Valid Folders (1:1 PDF + valid JSON): {valid_count}")
print(f"Issues Detected: {len(issues)}")

if issues:
    print("\nFirst 10 issues:")
    for issue in issues[:10]:
        print(" -", issue)
else:
    print("\nAll document folders follow the existing rules and naming conventions flawlessly!")
